<a id="unit-u.3f6a17523e7c4156b9b4ec21d8c5e904"></a>

<div>
  <div>02-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН НАҚТЫ УАҚЫТТАҒЫ АНАЛИТИКА</div>
  <div>Doris сканерлейтін деректерді азайту</div>
  <p>Сұрыптау кілттерін, Partition іріктеуін, bucket (tablet) іріктеуін, Query Profile сканерлеу дәлелдерін және қайталанатын кілт әрекетін салыстыру үшін сол 10,158,080 оқиға жолын пайдаланыңыз.</p>
  <span>Doris 4.1.3 · Сұрыптау кілті · Prefix Index · Partition · Tablet · Query Profile</span>
</div>

**2-модуль: Apache Doris архитектурасын тереңірек зерттеу** · [Сабақ](course.md) · [Тест](quiz02_doris_architecture_and_physical_design.ipynb)

Шамамен 25 минут жоспарлаңыз.

Бұл зертханалық жұмыс 1-зертханалық жұмысты жалғастырады. Ол `doris_course.events` кестесінен екі туынды ішкі кесте жасайды; S3-тен оқымайды және оған жазбайды. Әр үлкен кесте салыстыруы бірдей жолдарды пайдаланады, сондықтан сканерлеу ауқымының кішіреюі деректер жиынының кішіреюін емес, физикалық құрылымды көрсетеді. Ұяшықтарды ретімен орындаңыз.

<a id="unit-u.deb3cdb4d0d24b9880f83c5e64f619ff"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты бастапқы баптау

1-бөлімге дейін келесі ұяшықты орындаңыз. Ол ортақ көмекші модульді қайта жүктейді, 1-зертханалық жұмыстағы нәтиже стильдерін орнатады және кейінгі ұяшықтар пайдаланатын `lab` нысанын жасайды. Jupyter ядросын әр қайта іске қосқаннан кейін оны қайта орындаңыз. Ол Docker іске қоспайды және ешбір кестені өзгертпейді.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);

<a id="unit-u.e4bf5daefd0a41309336c68df04fe98d"></a>

<a id="1-establish-the-physical-design-baseline"></a>

## 1. Физикалық құрылымның бастапқы нұсқасын анықтау

Ноутбук бір түйінді біріктірілген Doris сынақ ортасы тоқтап тұрса оны іске қосады, қайта қосылады, `doris_course` таңдап, 1-зертханалық жұмыстың `events` кестесінің толық екенін тексереді. Бірінші нәтиже жол саны мен уақыт аралығын көрсетеді. Содан кейін `SHOW CREATE TABLE` 1-зертханалық жұмыс кілт пен тарату тармағын айқын бермегенде Doris не таңдағанын көрсетеді.

Осы курс ортасында бастапқы нұсқа Duplicate Key моделін, `(event_time, event_id, user_id)` сұрыптау кілтін және `RANDOM BUCKETS AUTO` қолданатын кесте болады. Нақты bucket саны 2-бөлімде тексеріледі.

Үш кілт бағаны автоматты анықталған; олар бастапқы кілт емес. Кесте моделі немесе кілт тармағы берілмесе, Doris әдетте Duplicate Key моделін таңдап, кілт бағандарын схеманың басындағы жарамды бағандардан шығарады. Бұл кілт бағандары сұрыптау кілтін, ал олардың бастапқы префиксі Prefix Index құрайды. Автоматты таңдау үш бағанға, 36 индекс байтына, Prefix Index қолдамайтын типке жеткенде немесе `VARCHAR` бағаны қосылғаннан кейін тоқтайды. Бұл схемада алғашқы үш бекітілген ұзындықты баған `(event_time, event_id, user_id)` болады.

Duplicate Key моделінде бұл бағандар сақтау ретін және Prefix Index анықтайды, бірақ қайталанатын кілт мәндері сақталады. Бұл жолды анықтап, қайталанатын кілт мәндерін қабылдамайтын MySQL `PRIMARY KEY` шектеуінен өзгеше. Әр кілтке бір логикалық жол және upsert әрекеті қажет болғанда Doris Unique Key моделін пайдаланады.

Анықтама: [Duplicate Key моделі](https://doris.apache.org/docs/4.x/table-design/data-model/duplicate/) · [CREATE TABLE](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/table-and-view/table/CREATE-TABLE/)

In [ ]:
lab.start_container()
lab.connect()
lab.execute("USE doris_course")
lab.check_prerequisites(2)

lab.sql("""
SELECT
    COUNT(*) AS row_count,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time
FROM events
""", title="Baseline data check")

lab.sql(
    "SHOW CREATE TABLE events",
    title="Baseline physical design",
    metadata_view="design",
);

<a id="unit-u.5a74d0a9472e461d90ac2a0ef44903dd"></a>

**Күтілетін нәтиже**

Жасыл белгі `events` ішінде 10,158,080 жол барын растайды; 1-зертханалық жұмыс аяқталмаған болса, ұяшық тоқтап, сол туралы хабарлайды. `Baseline data check` бір жол қайтарады:

| row_count | min_event_time | max_event_time |
|---:|---|---|
| 10,158,080 | 2019-12-01 00:00:12 | 2020-03-11 04:53:08 |

`Baseline physical design` ықшам нәтиже кестесінде көрсетілген сол алты құрылым таңдауын қайтарады:

| Құрылым таңдауы | Анықталған мән |
|---|---|
| Кесте моделі | Duplicate Key моделі |
| Сұрыптау кілті | event_time, event_id, user_id |
| Partition бөлу | Айқын Partition бөлу жоқ |
| Тарату | RANDOM |
| Әр Partition ішіндегі Bucket саны | AUTO |
| Реплика бөлу | Әр Tablet-ке 1 реплика · tag.location.default |

`AUTO` таңдаған нақты сан 2-бөлімде тексеріледі. Бастапқы кесте жоқ болса немесе жол саны сәйкес келмесе, жалғастырмас бұрын 1-зертханалық жұмысты аяқтаңыз.

<a id="unit-u.78dbe2998f154632a1b16c0c1c3d3cd7"></a>

<a id="2-map-the-table-to-its-physical-storage-hierarchy"></a>

## 2. Кестені физикалық сақтау иерархиясымен байланыстыру

Doris деректерге Catalog, Database және Table арқылы логикалық атау береді. Ішкі кестеде Partition әр Bucket үшін бір Tablet қамтиды. Жүктеу транзакциясы әр әсер етілген Tablet ішінде Rowset жасай алады, ал әр Rowset өзгермейтін бағандық Segment файлдарын қамтиды. Фондық Compaction оқу көлемінің ұлғаюын шектеу үшін шағын Rowset жиынтықтарын біріктіреді.

```text
internal Catalog
└── doris_course Database
    └── events Table
        └── Partition              logical range and lifecycle boundary
            └── Bucket / Tablet    distribution rule / physical data shard
                └── Rowset         version unit created by a load transaction or Compaction
                    └── Segment    immutable columnar data file
```

`SHOW PARTITIONS` Partition және Bucket метадеректерін көрсетеді. `SHOW TABLETS` Tablet нысандары мен репликаларын көрсетеді. `VersionCount` көрінетін дерек нұсқаларын көрсетеді; бұл Compaction үшін пайдалы дәлел, бірақ Rowset немесе Segment санының тікелей есебі емес.

1-бөлімдегі DDL bucket санының саясатын `AUTO` деп жазады. Төменде қайтарылған `Buckets` бағаны одан шыққан физикалық орналасуды жазады. Ағымдағы курс сынақ ортасында Doris бұл саясатты `events` үшін 10 bucket деп шешеді. Бұл әмбебап әдепкі мән емес, осы кесте мен ортада байқалған нәтиже.

`SHOW PARTITIONS` және `SHOW TABLETS` әр BE шамамен минутына бір рет FE түйініне хабарлайтын жол сандарын, өлшемдер мен нұсқа сандарын көрсетеді. Жүктеуден немесе қайта іске қосудан кейін бірден олар әлі `0` немесе `-1` көрсетеді, сондықтан келесі ұяшық алдымен Tablet нысандары кестенің барлық жолы туралы хабарлағанша күтеді. Бұл шамамен екі минутқа дейін созылуы мүмкін.

In [ ]:
lab.wait_for_tablet_rows("events", expected_rows=10_158_080)

lab.sql(
    "SHOW PARTITIONS FROM events",
    title="Baseline partitions",
    columns=["PartitionName", "State", "Buckets", "DistributionKey", "RowCount"],
)

lab.sql(
    "SHOW TABLETS FROM events ORDER BY LocalDataSize DESC LIMIT 10",
    title="Baseline tablets",
    columns=[
        "TabletId", "BackendId", "LocalDataSize",
        "RowCount", "VersionCount", "VisibleVersionCount",
    ],
);

<a id="unit-u.98c0231c7f284023bc45527f57af90e8"></a>

**Күтілетін нәтиже**

Жасыл белгі `events` кестесінің 10 Tablet нысаны мен бір Partition бөлімі барлық 10,158,080 жол туралы хабарлағанын көрсетеді. Одан кейін:

- Осы сынақ ортасында `events` кестесінде бір айқын бөлінбеген/әдепкі Partition және 10 Tablet бар.
- Әр Tablet-тің бір репликасы бар, өйткені All-in-One ортасында бір BE бар және бір реплика пайдаланылады.
- Tablet нысандары жолдардың әртүрлі ішкі жиындарын қамтиды, бірақ Random bucket бөлуі `user_id` теңдік предикаты бойынша іріктеуге болатын bucket кілтін бермейді.

<a id="unit-u.a079f798a1bc4f858ad916a42945252d"></a>

<a id="3-isolate-the-effect-of-the-leading-sort-key"></a>

## 3. Сұрыптау кілтінің бастапқы бағанының әсерін оқшаулау

`events_v2` бір Partition және 10 Random Bucket сақтайды, бірақ `user_id` бағанын сұрыптау кілтінің басына жылжытады. Бұл сұрыптау кілті мен Prefix Index әсерін оқшаулайды. Мұнда Duplicate Key моделінің кілт бағандары сақтау ретін анықтайды; олар бірегейлікті міндеттемейді.

Әр орындауда 1-зертханалық жұмыстың ішкі кестесінен сол 10,158,080 жол қайта құрылуы үшін кесте енгізуден бұрын тазартылады. Бұл салыстыруды қайталанатын етеді және жаңа физикалық орналасуды жазуға кеткен уақытты көрсетеді.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS events_v2 (
    user_id BIGINT NOT NULL,
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12, 2) NOT NULL DEFAULT "0.00"
)
DUPLICATE KEY(user_id, event_time, event_id)
DISTRIBUTED BY RANDOM BUCKETS 10
PROPERTIES (
    "replication_num" = "1"
)
""")

lab.execute("TRUNCATE TABLE events_v2")

lab.insert("""
INSERT INTO events_v2 (
    user_id, event_time, event_id, event_type, region, product_id, revenue
)
SELECT
    user_id, event_time, event_id, event_type, region, product_id, revenue
FROM events
""", title="Build the user-first sort layout")

lab.sql("""
SELECT 'events' AS table_name, COUNT(*) AS row_count, SUM(revenue) AS total_revenue
FROM events
UNION ALL
SELECT 'events_v2', COUNT(*), SUM(revenue)
FROM events_v2
ORDER BY table_name
""", title="v1 and v2 content check");

<a id="unit-u.9c269c954626483e912dec26fd833e41"></a>

**Күтілетін нәтиже**

Енгізу нәтижесі алдымен 10,158,080 әсер етілген жолды және `events_v2` құруға кеткен уақытты хабарлайды. Содан кейін екі тексеру жолы да `row_count = 10,158,080` және `total_revenue = 39,984,455.64` көрсетеді. Кесте мазмұндары бірдей; тек баған реті мен сұрыптау кілті өзгеше.

<a id="unit-u.e4fedc7dafa0407e9d621c9fc3ab8204"></a>

<a id="4-measure-how-the-leading-sort-key-reduces-scan-work"></a>

## 4. Сұрыптау кілтінің бастапқы бағаны сканерлеу жұмысын қалай азайтатынын өлшеу

Төрт сұрау екі бақыланатын жұп құрайды. Әр жұпта предикат пен агрегат бірдей, тек кесте өзгереді. Query Profile BE сканерлеу операторының орындалу кезіндегі дәлелдерін, мысалы қанша жол мен байт оқығанын тіркейді.

Әр Tablet ішінде Doris жолдарды сұрыптау кілті ретімен сақтап, кілттің бастапқы префиксі бойынша сирек Prefix Index құрады. Индекс сканерлеу қарастыруы тиіс жол ауқымдарын тарылтады; ол бірегейлікті міндеттемейді және өздігінен Partition не Tablet алып тастамайды.

- `events` `(event_time, event_id, user_id)` бойынша реттелген, сондықтан уақыт ауқымы оның кілтінің басына сәйкес келеді. `event_time` және `event_id` шектелмесе, `user_id` сүзгісі сол бастапқы префиксті пайдалана алмайды.
- `events_v2` `(user_id, event_time, event_id)` бойынша реттелген, сондықтан `user_id` теңдік сүзгісі оның кілтінің басына сәйкес келеді. Тек уақыт бойынша сүзгі бастапқы префикске енді сәйкес келмейді.

Алдымен кез келген MySQL клиентінде қолданар операторларыңызбен бір Query Profile өзіңіз алыңыз. Содан кейін көмекші құрал барлық төрт сұрау үшін сол қадамдарды қайталап, санағыштарын қатар көрсетеді.

<a id="unit-u.82c39437e2a148a692b13260f8368e63"></a>

<a id="capture-one-query-profile-by-hand"></a>

### Бір Query Profile қолмен алу

1. Осы сеанс үшін Profile жинауды қосып, қайталанатын сұрауға сканерлеусіз жауап беруі мүмкін үш кэшті өшіріңіз: SQL Cache, Query Cache және Condition Cache.
2. Сұрауды орындаңыз.
3. Бірден артынан `SELECT last_query_id()` орындаңыз. Ол жаңа ғана аяқталған сұраудың ID мәнін қайтарады; бұл сонымен бірге оның Profile ID мәні. Әр оператор осы мәнді ауыстырады, сондықтан оны сұраудан кейін бірден шақырыңыз.
4. `SHOW QUERY PROFILE` ең соңғы Profile жазбаларын тізеді. Өз ID мәніңіз бар жолды табыңыз.
5. Profile мәтінін оқыңыз. FE веб-интерфейсі оны төмендегі браузер сілтемесі арқылы **QueryProfile** бетінде көрсетеді (`root` пайдаланушысы, бос құпиясөз). FE HTTP API сол мәтінді қайтарады; екінші ұяшық оны `curl` арқылы оқиды.

In [ ]:
lab.show_browser_link(lab.fe_http_url, "Open Doris FE; select QueryProfile");

<a id="unit-u.ef8b0bbd2583450c82001fb2ede62418"></a>

**Күтілетін нәтиже:** браузерде ашуға арналған бір сілтеме пайда болады. Оны ашып, жұмыс істеп тұрған қызметке өтіңіз. Бұл зертханалық жұмыс қашықтағы хостта орындалса, сілтеме бапталған браузер мекенжайын пайдаланады; хост жағындағы API сұраулары ішкі мекенжайын сақтайды.

In [ ]:
lab.execute("SET enable_profile = true")
lab.execute("SET enable_sql_cache = false")
lab.execute("SET enable_query_cache = false")
lab.execute("SET enable_condition_cache = false")

lab.sql("""
SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM events_v2
WHERE user_id = 610871788
""", title="The query to profile")

profile_ids = lab.sql("SELECT last_query_id() AS query_id", title="Its query ID, which is also its Profile ID")
PROFILE_ID = profile_ids.iloc[0]["query_id"]

lab.sql(
    "SHOW QUERY PROFILE",
    title="Recent Query Profiles",
    columns=["Profile ID", "Task State", "Total", "Sql Statement"],
);

In [ ]:
lab.shell(f"""
# The BEs report their part after the query ends; read again until the scan appears.
for attempt in 1 2 3 4 5 6 7 8 9 10; do
  PROFILE=$(curl --silent --show-error -u root: "{lab.fe_http_url}/api/profile/text?query_id={PROFILE_ID}")
  echo "$PROFILE" | grep -q 'OLAP_SCAN_OPERATOR(' && break
  sleep 1
done
echo "$PROFILE" | sed -n '/MergedProfile/,/DetailProfile/p' \\
  | grep -E 'OLAP_SCAN_OPERATOR\\(|- (ScanRows|ScanBytes):'
""", title="Read the scan counters from the Profile text")

# Return the four settings to their defaults.
lab.execute("UNSET VARIABLE enable_profile")
lab.execute("UNSET VARIABLE enable_sql_cache")
lab.execute("UNSET VARIABLE enable_query_cache")
lab.execute("UNSET VARIABLE enable_condition_cache");

<a id="unit-u.94ef7858f13e46bb92a2fc3557a22f76"></a>

**Күтілетін нәтиже**

- Сұрау `event_count = 200` және `total_revenue = 24,192.36` қайтарады.
- `query_id` — бір дефисі бар оналтылық ID, мысалы `23e8b586a30c494a-b0b436918b654353`; сіздікі өзгеше болады.
- `Recent Query Profiles` ішінде сол ID, `Task State = OK` және сұрау мәтіні бар жол болады. Оның үстіндегі жол — `SELECT last_query_id()` операторының өзі: Profile қосылғанда әр оператордың өз жазбасы болады.
- Пәрмен бір `OLAP_SCAN_OPERATOR(... table_name=events_v2(events_v2) ...)` жолын, содан кейін оның `ScanBytes` және `ScanRows` мәндерін шығарады. Әр санағыш алдымен сканерлеудің орындау даналары бойынша қосындысын көрсетеді. Сыналған ортада `ScanRows` `sum 200` болды: `user_id` сұрыптау кілтінің басында болғандықтан, сканерлеу кестедегі 10,158,080 жолдың ішінен тек осы пайдаланушының 200 жолын оқыды. `ScanBytes` — сол жолдар үшін сканерлеу оқыған деректер.

`sed` және `grep` тек MergedProfile сканерлеу санағыштарын қалдырады; толық мәтінде әр орындау данасына бір бөлімнен тұратын DetailProfile да бар. Төрт `UNSET VARIABLE` операторы сеансты әдепкі мәндерге қайтарады.

<a id="unit-u.45c90f5877d9427e9202db3ac0d7b839"></a>

<a id="compare-the-four-queries"></a>

### Төрт сұрауды салыстыру

Келесі ұяшық төрт сұраудың әрқайсысына бірдей қадамдарды орындайды: бірдей сеанс айнымалыларын орнатады, сұрауды орындайды, `last_query_id()` оқиды және FE HTTP API арқылы Profile мәтінін алады. Содан кейін алдыңғы баптауларыңызды қалпына келтіреді. Бір жиналмалы панельде орындалған барлық оператор, екіншісінде әр Profile үшін `curl` пәрмені тізіледі.

Profile бағандарын былай оқыңыз:

| Метрика | Осы салыстырудағы мағынасы |
|---|---|
| `event_count`, `total_revenue` | Сұрау нәтижесі; сәйкес мәндер мағыналық баламалылықты дәлелдейді. |
| `rows_read` | Сканерлеу жағындағы сүзуден кейін сканерлеу қайтарған жолдар. Әр сұрау жұбы ішінде бірдей қалуы тиіс. |
| `scan_rows` | Сақтау орнынан қарастырылған жолдар. Бұл — сұрыптау кілтін салыстыруға арналған негізгі жол көлемі дәлелі. |
| `scan_bytes` | Сканерлеу қарастырған байттар. Ол `scan_rows` дәлелін толықтырады, бірақ кодтау, таңдалған бағандар және Compaction нақты мәнге әсер етуі мүмкін. |

Нақты сканерлеу көлемі Compaction кейін және әр компьютерде өзгеруі мүмкін. Бірдей байт санын күтудің орнына жұпталған мәндердің бағыты мен ауқымын салыстырыңыз.

In [ ]:
lab.compare_profiles([
    ("events · user filter", """
        SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events
        WHERE user_id = 610871788
    """),
    ("events_v2 · user filter", """
        SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events_v2
        WHERE user_id = 610871788
    """),
    ("events · time filter", """
        SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events
        WHERE event_time >= '2020-03-01 00:00:00'
          AND event_time <  '2020-03-02 00:00:00'
    """),
    ("events_v2 · time filter", """
        SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events_v2
        WHERE event_time >= '2020-03-01 00:00:00'
          AND event_time <  '2020-03-02 00:00:00'
    """),
], title="Sort-key runtime comparison");

<a id="unit-u.15767dfe53764217998882f284d223b3"></a>

**Күтілетін нәтиже**

| Сұрау | Күтілетін `event_count` | Күтілетін `total_revenue` | Күтілетін `rows_read` | Сканерлеуден күтілетін бақылау |
|---|---:|---:|---:|---|
| `events · user filter` | 200 | 24,192.36 | 200 | `user_id` кілттің басында болмағандықтан, `scan_rows` жоғары болып қалады. |
| `events_v2 · user filter` | 200 | 24,192.36 | 200 | `user_id` кілттің басында болғандықтан, `scan_rows` едәуір төмен. |
| `events · time filter` | 75,259 | 5,670,241.29 | 75,259 | `event_time` кілттің басында болғандықтан, `scan_rows` төмен. |
| `events_v2 · time filter` | 75,259 | 5,670,241.29 | 75,259 | `event_time` енді кілттің басында болмағандықтан, `scan_rows` едәуір жоғары. |

Әр жұптың нәтижесі мен `rows_read` мәні сәйкес келуі тиіс. Айырмашылық `scan_rows` және `scan_bytes` мәндерінде көрінеді: бір қатынау үлгісін сұрыптау кілтінің басына орналастыру басқа қатынау үлгісінің тиімділігін төмендетуі мүмкін.

<a id="unit-u.e68aad6174064f788217cbab62a04262"></a>

<a id="5-enable-partition-pruning-and-bucket-tablet-pruning"></a>

## 5. Partition іріктеуі мен bucket (tablet) іріктеуін қосу

`events_v3` бастапқы сұрыптау кілтін сақтайды, `event_time` бойынша күндік Auto Partitioning қосады және әр Partition бөлімін `HASH(user_id)` арқылы таратады. Енгізілген жолдарға сол күн алғаш қажет болғанда Doris бөлім жасайды. Дереккөзде 26 әртүрлі күннің оқиғалары бар, сондықтан дайын кестеде 26 Partition және 260 Tablet болуы тиіс: әр бөлімде 10 Tablet.

Әр орындағанда Auto Partitioning пен толық физикалық орналасу өзгермеген бастапқы кестеден қайта құрылуы үшін кесте енгізуден бұрын тазартылады. Сондықтан көрсетілген енгізу ұзақтығы қайта орындауды өткізіп жіберуді емес, нақты жазуды өлшейді.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS events_v3 (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12, 2) NOT NULL DEFAULT "0.00"
)
DUPLICATE KEY(event_time, event_id, user_id)
AUTO PARTITION BY RANGE (date_trunc(event_time, 'day'))
()
DISTRIBUTED BY HASH(user_id) BUCKETS 10
PROPERTIES (
    "replication_num" = "1"
)
""")

lab.execute("TRUNCATE TABLE events_v3")

lab.insert("""
INSERT INTO events_v3 (
    event_time, event_id, user_id, event_type, region, product_id, revenue
)
SELECT
    event_time, event_id, user_id, event_type, region, product_id, revenue
FROM events
""", title="Build the partitioned and hash-bucketed layout")

lab.sql("""
SELECT 'events' AS table_name, COUNT(*) AS row_count, SUM(revenue) AS total_revenue
FROM events
UNION ALL
SELECT 'events_v3', COUNT(*), SUM(revenue)
FROM events_v3
ORDER BY table_name
""", title="v1 and v3 content check");

<a id="unit-u.6df689ef01674bb3839ee86ed290f2fb"></a>

**Күтілетін нәтиже**

Енгізу нәтижесі алдымен 10,158,080 әсер етілген жолды және `events_v3` құруға кеткен уақытты хабарлайды. Содан кейін екі кестеде де 10,158,080 жол және 39,984,455.64 жалпы түсім болады. Auto Partitioning логикалық деректер жиынын емес, орналасуды өзгертеді.

<a id="unit-u.cf7cfdf6bfe7445fa8b7bfc822be08fe"></a>

<a id="6-connect-load-transactions-to-tablet-storage-and-compaction"></a>

## 6. Жүктеу транзакцияларын Tablet сақтауы және Compaction үдерісімен байланыстыру

`SHOW PARTITIONS` және `SHOW TABLETS` әр BE шамамен минутына бір рет FE түйініне хабарлайтын жол сандарын, өлшемдер мен нұсқа сандарын көрсетеді. Жүктеуден немесе қайта іске қосудан кейін бірден олар әлі `0` немесе `-1` көрсетеді, сондықтан келесі ұяшық алдымен Tablet нысандары кестенің барлық жолы туралы хабарлағанша күтеді. Бұл шамамен екі минутқа дейін созылуы мүмкін.

Содан кейін кесте анықтамасын оның физикалық орналасуымен байланыстыру үшін үш Doris метадерек операторын пайдаланыңыз. Әр оператор код ұяшығында көрініп тұрады, ал оның нәтижесі тиісті метадеректердің ықшам көрінісін береді.

- Құрылым қорытындысы кесте моделін, сұрыптау кілтін, Partition ережесін, тарату ережесін, bucket санын және реплика бөлуді шығарады.
- Partition қорытындысы Auto Partitioning оқиға күндерінен нақты не жасағанын тексереді.
- Tablet қорытындысы барлық Tablet нысандарын қамтып, еркін он жолдық үлгі таңдаудың орнына ауқымдар мен жиынтықтарды көрсетеді.

Tablet қорытындысында `Unique tablets` физикалық деректер шардтарын санайды, `Replicas per tablet` сынақ ортасының репликация коэффициентін растайды, `Rows per tablet` таралу ауқымын көрсетеді, ал `Local replica size` жергілікті репликалар алатын сақтау орнын қосады.

Жүктеу транзакциясы әсер етілген Tablet нысандарының ішінде өзгермейтін Rowset жариялайды, ал әр Rowset бір немесе бірнеше бағандық Segment қамтиды. Кейін фондық Compaction сұрау нәтижелерін өзгертпей бір Tablet ішіндегі Rowset жиынтықтарын біріктіреді. `version count` — пайдалану кезіндегі пайдалы дәлел, бірақ ол Rowset немесе Segment санының тікелей есебі де, бекітілген күтілетін мән де емес.

In [ ]:
lab.wait_for_tablet_rows("events_v3", expected_rows=10_158_080)

lab.sql(
    "SHOW CREATE TABLE events_v3",
    title="v3 physical design",
    metadata_view="design",
)

lab.sql(
    "SHOW PARTITIONS FROM events_v3 ORDER BY PartitionName",
    title="v3 partition layout",
    metadata_view="partitions",
)

lab.sql(
    "SHOW TABLETS FROM events_v3",
    title="v3 tablet and version overview",
    metadata_view="tablets",
);

<a id="unit-u.42fdd7f7d5cb4aa687f5259da7012267"></a>

**Күтілетін нәтиже**

Жасыл белгі `events_v3` кестесінің 260 Tablet нысаны мен 26 Partition бөлімі барлық 10,158,080 жол туралы хабарлағанын көрсетеді. Одан кейін:

- Құрылым қорытындысы Duplicate Key моделін, `(event_time, event_id, user_id)` сұрыптау кілтін, күндік Auto Range бөлуді, `HASH(user_id)`, әр Partition ішіндегі 10 Bucket және бір репликаны көрсетеді.
- Partition қорытындысы дұрыс жұмыс істейтін 26 күндік бөлімді, жалпы 260 Tablet пен 10,158,080 жолды көрсетеді. Шекаралары 2019-12-01 күнінен кірмейтін жоғарғы шек 2020-03-12 күніне дейін созылады; Doris тек бастапқы деректерде бар күндер үшін бөлімдер жасайды.
- Tablet қорытындысы 260 бірегей Tablet пен әр Tablet-ке бір реплика көрсетеді. Ол еркін таңдалған ішкі жиынды көрсетудің орнына барлық Tablet-ті қорытындылайды.
- Жергілікті өлшем, әр Tablet-тегі жол саны және `VersionCount` тарату мен фондық Compaction әсерінен өзгеруі мүмкін. Олардың ауқымдарын түсіндіріңіз; нақты мәндерді бағалау өлшемі етпеңіз.

<a id="unit-u.abf2a00e2e614bdc8d542c5fa5fb1cfc"></a>

<a id="7-prove-that-fe-pruning-reduces-the-scan-scope"></a>

## 7. FE іріктеуі сканерлеу ауқымын азайтатынын дәлелдеу

Предикатты төменге итеру мен іріктеу әртүрлі сұрақтарға жауап береді. Төменге итерілген предикатты BE сканерлеу операторы сақтау орнына жақын есептейді. Partition іріктеуі мен bucket (tablet) іріктеуі бұдан бұрын орындалады: FE кесте метадеректері арқылы физикалық нысандарды үлестірілген сұрау жоспарынан алып тастайды.

Келесі ұяшық `events` және `events_v3` кестелеріндегі бірдей үш предикат пішінін салыстырады. Нәтиже кестесі `partitions` және `tablets` үшін тек таңдалған/жалпы қатынастарын қалдырады; жасалған Partition атаулары мен толық жоспарлар жиналған нәтижеде қолжетімді болып қалады.

In [ ]:
lab.compare_explain([
    ("events · time", """
        EXPLAIN SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events
        WHERE event_time >= '2020-03-01 00:00:00'
          AND event_time <  '2020-03-02 00:00:00'
    """),
    ("events_v3 · time", """
        EXPLAIN SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events_v3
        WHERE event_time >= '2020-03-01 00:00:00'
          AND event_time <  '2020-03-02 00:00:00'
    """),
    ("events · user", """
        EXPLAIN SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events
        WHERE user_id = 610871788
    """),
    ("events_v3 · user", """
        EXPLAIN SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events_v3
        WHERE user_id = 610871788
    """),
    ("events · time + user", """
        EXPLAIN SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events
        WHERE event_time >= '2020-03-01 00:00:00'
          AND event_time <  '2020-03-02 00:00:00'
          AND user_id = 610871788
    """),
    ("events_v3 · time + user", """
        EXPLAIN SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events_v3
        WHERE event_time >= '2020-03-01 00:00:00'
          AND event_time <  '2020-03-02 00:00:00'
          AND user_id = 610871788
    """),
], title="Partition pruning and bucket (tablet) pruning");

<a id="unit-u.0e802f0c58f04d8aa84057f3c481ad45"></a>

**Күтілетін нәтиже**

| Предикат | Бастапқы `events` | `events_v3` |
|---|---|---|
| Тек уақыт | `partitions=1/1`, `tablets=10/10` | `partitions=1/26`, `tablets=10/10`: таңдалған күннің барлық 10 Tablet нысаны |
| Тек пайдаланушы | `partitions=1/1`, `tablets=10/10` | `partitions=26/26`, `tablets=26/260`: әр Partition ішінде бір Hash Tablet |
| Уақыт + пайдаланушы | `partitions=1/1`, `tablets=10/10` | `partitions=1/26`, `tablets=1/10`: бір Partition және бір Tablet |

`tablets` қатынасының бөлімі тек таңдалған Partition ішіндегі Tablet нысандарын есептейді: бір күн үшін 10, барлық 26 күн үшін 260. Алымы бөлімінен кіші болса, бұл іріктеуді дәлелдейді. Барлық алты жоспарда тиісті сканерлеу предикаты әлі де көрсетілуі тиіс; ол іріктеуге тәуелсіз предикатты төменге итеруді көрсетеді.

<a id="unit-u.30a0ea53435b49c7a6b1677551c5669d"></a>

<a id="8-verify-that-pruning-does-not-change-query-results"></a>

## 8. Іріктеу сұрау нәтижелерін өзгертпейтінін тексеру

7-бөлім жоспарланған Partition және Bucket/Tablet сканерлеу ауқымын салыстыру үшін `EXPLAIN` пайдаланды. `EXPLAIN` сұрауды орындамайды және оның талдамалық нәтижесін қайтармайды.

Сондықтан бұл бөлім біріктірілген сүзгісі бар `SELECT` сұрауын екі кестеде де орындайды. Екі нәтиже жолы бірдей болуы тиіс: `events_v3` деректерді немесе сұрау мағынасын емес, Doris деректерді қай жерден оқитынын өзгертеді.

In [ ]:
lab.sql("""
SELECT 'events' AS table_name, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM events
WHERE event_time >= '2020-03-01 00:00:00'
  AND event_time <  '2020-03-02 00:00:00'
  AND user_id = 610871788
UNION ALL
SELECT 'events_v3', COUNT(*), SUM(revenue)
FROM events_v3
WHERE event_time >= '2020-03-01 00:00:00'
  AND event_time <  '2020-03-02 00:00:00'
  AND user_id = 610871788
ORDER BY table_name
""", title="Combined-filter result check");

<a id="unit-u.ba765498ecba4ffcab9a3bfbbc6cca80"></a>

**Күтілетін нәтиже**

Екі жол да 11 оқиғаны және 4,592.14 жалпы түсімді көрсетеді. `events_v3` сол нәтижеге жоспарланған сканерлеу ауқымы кішірек болғанда жетеді.

<a id="unit-u.0fe90c41dfd845859b3152b5cfe34cf1"></a>

<a id="9-explain-repeated-key-behavior-with-table-models"></a>

## 9. Кесте модельдері арқылы қайталанатын кілт әрекетін түсіндіру

Бұл шағын кестелер бірдей екі жазу транзакциясын қабылдайды. Біріншісі 1001 және 1002 пайдаланушылары үшін бір-бір жол жазады. Екіншісі бар `(1001, 2026-01-01)` кілтіне тағы бір жол жазады. Кірісті бірдей ұстау тағы бір үлкен импортсыз модель әрекетін көрсетеді.

- Duplicate Key моделі барлық егжей-тегжейлі жолды сақтайды.
- Unique Key моделі кілттің соңғы жолын көрсетеді.
- Aggregate Key моделі мән бағандарын жарияланған агрегаттау функциясына қарай біріктіреді; бұл мысал `SUM` пайдаланады.

`model_agg` үшін `revenue DECIMAL(12, 2) SUM` агрегат кілт мәндері бірдей жолдардың қалай біріктірілетінін жариялайды. Сондықтан тікелей таңдалғанда Doris әр `(user_id, event_date)` үшін бір логикалық `revenue` мәнін көрсетеді. Соңғы салыстыру осы логикалық жолдарға айқын `SUM(revenue) GROUP BY user_id, event_date` орындап, екі сұрау пішінін қатар көрсетеді.

In [ ]:
for table_name in ("model_dup", "model_uniq", "model_agg"):
    lab.execute(f"DROP TABLE IF EXISTS {table_name}")

lab.execute("""
CREATE TABLE model_dup (
    user_id BIGINT NOT NULL,
    event_date DATE NOT NULL,
    revenue DECIMAL(12, 2) NOT NULL
)
DUPLICATE KEY(user_id, event_date)
DISTRIBUTED BY HASH(user_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("""
CREATE TABLE model_uniq (
    user_id BIGINT NOT NULL,
    event_date DATE NOT NULL,
    revenue DECIMAL(12, 2) NOT NULL
)
UNIQUE KEY(user_id, event_date)
DISTRIBUTED BY HASH(user_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("""
CREATE TABLE model_agg (
    user_id BIGINT NOT NULL,
    event_date DATE NOT NULL,
    revenue DECIMAL(12, 2) SUM NOT NULL
)
AGGREGATE KEY(user_id, event_date)
DISTRIBUTED BY HASH(user_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

for table_name in ("model_dup", "model_uniq", "model_agg"):
    lab.execute(f"""
        INSERT INTO {table_name} VALUES
            (1001, '2026-01-01', 10.00),
            (1002, '2026-01-01', 20.00)
    """)
    lab.execute(f"INSERT INTO {table_name} VALUES (1001, '2026-01-01', 15.00)")

lab.sql("""
SELECT 'duplicate' AS model, user_id, event_date, revenue
FROM model_dup
UNION ALL
SELECT 'unique', user_id, event_date, revenue
FROM model_uniq
UNION ALL
SELECT 'aggregate', user_id, event_date, revenue
FROM model_agg
ORDER BY model, user_id, revenue
""", title="Repeated-key results")

lab.sql("""
SELECT
    'Direct model value' AS query_form,
    user_id,
    event_date,
    revenue AS revenue_result
FROM model_agg

UNION ALL

SELECT
    'Explicit SUM with GROUP BY' AS query_form,
    user_id,
    event_date,
    SUM(revenue) AS revenue_result
FROM model_agg
GROUP BY user_id, event_date

ORDER BY user_id, query_form
""", title="Aggregate Key model query equivalence");

<a id="unit-u.7608819db1184b318e740a19cce71bd7"></a>

**Күтілетін нәтиже**

`Repeated-key results` бірдей жазулардың әртүрлі логикалық нәтиже беретінін көрсетеді:

| Модель | 1001 пайдаланушысының нәтижесі | Барлық көрінетін жол |
|---|---|---:|
| Duplicate Key моделі | екі жол: 10.00 және 15.00 | 3 |
| Unique Key моделі | соңғы мән: 15.00 | 2 |
| Aggregate Key моделі | қосынды мән: 25.00 | 2 |

`Aggregate Key model query equivalence` әр агрегат кілті үшін сұраудың екі пішінін көрсетеді:

| query_form | user_id | event_date | revenue_result |
|---|---:|---|---:|
| Direct model value | 1001 | 2026-01-01 | 25.00 |
| Explicit SUM with GROUP BY | 1001 | 2026-01-01 | 25.00 |
| Direct model value | 1002 | 2026-01-01 | 20.00 |
| Explicit SUM with GROUP BY | 1002 | 2026-01-01 | 20.00 |

Мәндер сәйкес келеді, өйткені сұрау толық агрегат кілті бойынша топтайды және `revenue` үшін жарияланған сол `SUM` функциясын қолданады. Doris Aggregate Key моделінің логикалық жолдарын жасағанда қайталанатын кілт мәндерін әлдеқашан біріктіреді; айқын сұрау сол логикалық жолдарды қайта агрегаттап, осы нәтижені өзгертпейді.

Бұл баламалылық тек осы топтау мен агрегаттауға тән. Басқа кілт жиыны бойынша топтайтын немесе басқа метриканы есептейтін сұрауға өзінің агрегаттауы әлі де қажет. Өндірістік көлемде Aggregate Key моделі көптеген бастапқы жолдың кілттері бірдей болғанда ең пайдалы, өйткені бұл жолдарды саны азырақ агрегатталған логикалық жолдармен көрсетуге болады.

<a id="unit-u.84796536055f424fa95d2f478ed24248"></a>

<a id="10-practice-predict-what-a-region-filter-can-prune"></a>

## 10. Жаттығу: аймақ сүзгісі нені іріктей алатынын болжау

`events_v3` күн бойынша Partition бөлімдеріне, ал `HASH(user_id)` бойынша bucket-терге бөлінген. Ештеңе орындамас бұрын `events_v3` кестесіндегі екі сүзгі үшін `EXPLAIN` көрсетуі тиіс деп күткен `partitions` және `tablets` қатынастарын жазып алыңыз:

1. `WHERE region = 'region_03'`
2. `WHERE region = 'region_03'` және бір күн: `2020-03-01 00:00:00` мезетін қоса алғанда, `2020-03-02 00:00:00` мезетін қоспағанда

Содан кейін 7-бөлімдегідей `lab.compare_explain` арқылы болжамыңызды тексеріңіз және екінші сүзгіні сұрау ретінде орындап, қанша оқиға мен түсімді қамтитынын көріңіз.

In [ ]:
# Your code here

<a id="unit-u.d72b1553af0646df9eb6b803f33b10c8"></a>

Келесі ұяшық — үлгі шешім; ол жиналған. Салыстыру үшін оны ашыңыз немесе орындаңыз.

In [ ]:
region_query = """
SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM events_v3
WHERE region = 'region_03'
"""
region_day_query = region_query + """
  AND event_time >= '2020-03-01 00:00:00'
  AND event_time <  '2020-03-02 00:00:00'
"""
lab.compare_explain([
    ("events_v3 · region", "EXPLAIN " + region_query),
    ("events_v3 · day + region", "EXPLAIN " + region_day_query),
], title="What a region filter can prune")
lab.sql(region_day_query, title="Events of region_03 on 2020-03-01");

<a id="unit-u.721b643312a84a228fc36fb4bd7310d6"></a>

**Күтілетін нәтиже**

| Сүзгі | `partitions` | `tablets` |
|---|---|---|
| Тек аймақ | 26/26 | 260/260 |
| Күн және аймақ | 1/26 | 10/10 |

`region` Partition бағаны да, Hash кілті де емес, сондықтан аймақ сүзгісі ештеңені алып тастамайды: жоспар барлық Partition және барлық Tablet нысанын оқып, сканерлеуде `region = 'region_03'` қолданады. Күн 26 Partition ішінен 25-ін алып тастайды, бірақ аймақ қалған күннің он Tablet нысанын әлі тарылта алмайды. Сұрау **9,640** оқиға және **679,830.92** түсім қайтарады.

Аймақ бойынша іріктеу үшін Partition өткізіп жіберетін Partition кілтінде немесе Tablet өткізіп жіберетін Hash кілті ретінде `region` қажет; 4-модуль неліктен тек сегіз мәні бар бағанның Hash кілтіне нашар келетінін түсіндіреді. Іріктей алмайтын сүзгі де деректерге жақын, сканерлеу ішінде орындалады; бұл — предикатты төменге итеру.

<a id="unit-u.ad96b13016a54916952535d7e1325490"></a>

<a id="stop-the-doris-sandbox"></a>

### Doris сынақ ортасын тоқтату

Жұмысты аяқтап, контейнердің CPU мен жадын босатқыңыз келгенде осы қосымша ұяшықты орындаңыз. Ол Doris үдерістерін тоқтатады, бірақ контейнерді, образды, FE метадерек томын және BE сақтау томын сақтайды. Сынақ ортасын пайдалануды жалғастырғыңыз келсе, оны өткізіп жіберіңіз.

In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris-rta
docker inspect --format 'container={{.State.Status}}' doris-rta
""", title="Stop the Doris sandbox");

<a id="unit-u.1651058c5d254680a1ef47ed9f2ae666"></a>

**Күтілетін нәтиже:** Docker `container=exited` деп хабарлайды. Бастапқы және одан жасалған кестелер атаулы томдарда қалады.

<a id="restart-the-doris-sandbox"></a>

### Doris сынақ ортасын қайта іске қосу

Сол 2-зертханалық жұмыс кестелерімен жалғастырғыңыз келгенде осы ұяшықты орындаңыз. Ол бар контейнерді іске қосады, Doris дұрыс жұмыс істейтін күйге жеткенше күтеді, FE түйініне қайта қосылып, сақталған жол сандарын тексереді.

Бұл қайта іске қосу ұяшығы идемпотентті: оны Docker Desktop пен контейнер тоқтаған, іске қосылып жатқан немесе жұмыс істеп тұрған кезде орындауға болады. macOS жүйесінде қажет болса Docker Desktop ашылады; Linux жүйесінде ұяшықты орындамас бұрын Docker Engine іске қосыңыз.


In [ ]:
lab.start_container()

lab.connect()
lab.execute("USE doris_course")
lab.sql("""
SELECT 'events' AS table_name, COUNT(*) AS recovered_rows FROM events
UNION ALL
SELECT 'events_v2', COUNT(*) FROM events_v2
UNION ALL
SELECT 'events_v3', COUNT(*) FROM events_v3
ORDER BY table_name
""", title="Recovered Lab 2 tables");

<a id="unit-u.85f34dd49a1543cdac6f2db467142fea"></a>

**Күтілетін нәтиже:** контейнер `healthy` күйіне оралады және үш кестенің бәрі **10,158,080** жол көрсетеді.

<a id="unit-u.f85ab349f1934096b446db6be57660be"></a>

<a id="lab-complete"></a>

## Зертханалық жұмыс аяқталды

Физикалық құрылымды өзгерте отырып, бір логикалық деректер жиынын тұрақты ұстадыңыз. Query Profile сұрыптау кілтінің бастапқы бағанын BE сканерлеу жұмысымен байланыстырды; `EXPLAIN` уақыт бөлімдері мен Hash bucket-терін FE іріктеуімен байланыстырды; сақтау метадеректері жүктеу транзакцияларын Tablet, нұсқалар, Segment және Compaction ұғымдарымен байланыстырды; ал соңғы сынақ деректері әр кесте моделінің қайталанатын кілттерді қалай өңдейтінін көрсетті.

Негізгі айырмашылықтар:

- Сұрыптау кілті мен Prefix Index сканерлеуге таңдалған Tablet ішінен тиісті кілт ауқымдарын табуға көмектеседі.
- Partition шекарасы FE түйініне бөлімдерді алып тастауға мүмкіндік береді.
- Hash тарату кілті FE түйініне үйлесімді теңдік предикаттары үшін Tablet нысандарын алып тастауға мүмкіндік береді.
- Предикатты төменге итеру сүзгілерді сканерлеу операторында есептейді; оның өзі Partition немесе Tablet алып тасталды дегенді білдірмейді.
- Duplicate Key, Unique Key және Aggregate Key модельдері қайталанатын кілттердің мағынасын басқарады.

Ресми анықтамалар: [Duplicate Key моделі](https://doris.apache.org/docs/4.x/table-design/data-model/duplicate/) · [Auto Partitioning](https://doris.apache.org/docs/4.x/table-design/data-partitioning/auto-partitioning/) · [EXPLAIN](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/data-query/EXPLAIN/) · [Query Profile](https://doris.apache.org/docs/4.x/query-acceleration/query-profile/) · [Compaction](https://doris.apache.org/docs/4.x/key-features/data-compaction/)

<a id="unit-u.c631c9971dd54593a3750a7045fee342"></a>

<a id="next"></a>

## Келесі қадам

[2-тест: Apache Doris архитектурасын тереңірек зерттеу](quiz02_doris_architecture_and_physical_design.ipynb) ашыңыз, содан кейін [3-модуль: Apache Doris жүйесіне деректерді жүктеу](../module03-loading-data/course.md) бөліміне өтіңіз.